In [8]:
# 연도x중분류 별 상위 70% 추출하기 전 키워드가 없는 논문 제거

import json
import pandas as pd
import os

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
input_json_path = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD.json'
output_json_path = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD_Filtered.json'

# =============================================================================
# 2. 데이터 로드 및 필터링
# =============================================================================
print(f"📂 파일 로드 중: {input_json_path}")

if os.path.exists(input_json_path):
    # 1) JSON 로드
    with open(input_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    if "NODE_LIST" not in data:
        print("❌ 오류: 'NODE_LIST' 키를 찾을 수 없습니다.")
        exit()

    # 2) DataFrame 변환
    df = pd.DataFrame(data["NODE_LIST"])
    original_count = len(df)
    
    # 3) 필터링 로직 적용
    # 'KYWD' 컬럼이 아예 없는 경우를 대비
    if 'KYWD' in df.columns:
        # NaN 값을 빈 문자열로 치환 후, 앞뒤 공백 제거
        df['KYWD'] = df['KYWD'].fillna("").astype(str).str.strip()
        
        # 키워드가 빈 문자열("")이 아닌 것만 남김
        df_filtered = df[df['KYWD'] != ""]
    else:
        # KYWD 컬럼 자체가 없으면 모든 데이터가 대상이 아님 (모두 삭제됨)
        print("⚠️ 경고: 데이터에 'KYWD' 필드가 존재하지 않습니다. 모든 데이터가 삭제됩니다.")
        df_filtered = pd.DataFrame(columns=df.columns)

    filtered_count = len(df_filtered)
    removed_count = original_count - filtered_count

    # =============================================================================
    # 3. 결과 저장
    # =============================================================================
    # JSON 구조 생성
    result_data = {"NODE_LIST": df_filtered.to_dict(orient='records')}

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(result_data, f, ensure_ascii=False, indent=4)

    print("-" * 50)
    print(f"✅ 작업 완료!")
    print(f"   - 원본 데이터 수: {original_count}건")
    print(f"   - 삭제된 데이터 수: {removed_count}건 (키워드 없음/공백)")
    print(f"   - 남은 데이터 수: {filtered_count}건")
    print(f"   - 저장 파일명: {output_json_path}")
    print("-" * 50)
    
else:
    print(f"❌ 오류: 입력 파일({input_json_path})이 존재하지 않습니다.")

📂 파일 로드 중: SSU_Datathon2025_공학분야_62199_Increase_KYWD.json
--------------------------------------------------
✅ 작업 완료!
   - 원본 데이터 수: 62199건
   - 삭제된 데이터 수: 11036건 (키워드 없음/공백)
   - 남은 데이터 수: 51163건
   - 저장 파일명: SSU_Datathon2025_공학분야_62199_Increase_KYWD_Filtered.json
--------------------------------------------------


In [9]:
# 연도x중분류 별 상위 70% 씩 추출

import json
import pandas as pd
import os
import re

# -----------------------------------------------------------
# 1. 파일 경로 설정
# -----------------------------------------------------------
json_file_path = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD_Filtered.json'

file_paths_if = {
    2021: '../2021_인용지수_2년분.xls',
    2022: '../2022_인용지수_2년분.xls',
    2023: '../2023_인용지수_2년분.xls',
    2024: '../2024_인용지수_2년분.xls'
}

# -----------------------------------------------------------
# 2. 사용자 수기 확인 내역 (매핑 테이블) 적용
# -----------------------------------------------------------
manual_mapping = {
    "(사)한국CDE학회": "한국CDE학회",
    "ICT플랫폼학회": "아이씨티플랫폼학회",
    "유공압건설기계학회": "사단법인 유공압건설기계학회",
    "한국로봇학회(논문지)": "한국로봇학회",
    "한국염색가공학회": "한국염색가공학회",
    "한국위험물학회": "한국위험물학회",
    "한국자동차안전학회": "사단법인 한국자동차안전학회",
    "한국전자파학회JEES": "한국전자파학회",
    "한국정보통신학회JICCE": "한국정보통신학회",
    "한국컴퓨터그래픽스학회": "(사)한국컴퓨터그래픽스학회",
    "한국콘텐츠학회(IJOC)": "한국콘텐츠학회",
    "한국환경에너지공학회": "(사)한국환경에너지공학회"
}

# -----------------------------------------------------------
# 3. 유틸리티 함수
# -----------------------------------------------------------
def normalize_name(name):
    """매칭 확률을 높이기 위해 공백과 특수문자를 제거"""
    if pd.isna(name): return ""
    return re.sub(r'[^a-zA-Z0-9가-힣]', '', str(name).upper())

def load_if_database(file_paths):
    """인용지수 DB 구축"""
    db = {}
    print("📂 인용지수 데이터베이스 구축 중...")
    
    for year, path in file_paths.items():
        if not os.path.exists(path):
            db[year] = {}
            continue
            
        try:
            df = pd.read_excel(path, engine='xlrd')
            df.columns = df.columns.str.replace('\n', '').str.strip()
            
            # IF 컬럼 찾기
            if_cols = [c for c in df.columns if '2년' in c and 'IF' in c]
            if not if_cols:
                db[year] = {}
                continue
            if_col = if_cols[0]
            
            # 발행기관 컬럼 찾기
            org_cols = [c for c in df.columns if any(x in c for x in ['발행기관', '발행처', '학회'])]
            target_col = org_cols[0] if org_cols else (df.columns[1] if len(df.columns) > 1 else df.columns[0])

            # 점수 매핑
            mapping = {}
            df[if_col] = pd.to_numeric(df[if_col], errors='coerce').fillna(0)
            
            for idx, row in df.iterrows():
                org_name = row[target_col]
                score = row[if_col]
                norm_name = normalize_name(org_name)
                
                if norm_name:
                    mapping[norm_name] = max(mapping.get(norm_name, 0), score)
            
            db[year] = mapping
            
        except Exception as e:
            print(f"   ❌ {year}년 로드 실패: {e}")
            db[year] = {}
            
    db[2025] = db.get(2024, {})
    return db

def main():
    # 1. 인용지수 DB 로드
    if_db = load_if_database(file_paths_if)
    
    # 2. JSON 논문 데이터 로드
    print(f"📂 논문 데이터 로드 중... ({json_file_path})")
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        if "NODE_LIST" not in data:
            print("오류: NODE_LIST 키가 없습니다.")
            return

        df = pd.DataFrame(data["NODE_LIST"])
        
        # 분석을 위해 임시로 Year 컬럼 생성
        df['temp_Year'] = df['PBSH'].astype(str).str.strip().str[:4]
        
        # 3. IF 점수 매핑 (보정 로직 적용)
        print("📊 보정된 매핑 테이블을 사용하여 IF 점수 계산 중...")
        
        def get_score(row):
            try:
                y = int(row['temp_Year'])
            except:
                y = 0
            
            original_name = row['IPRD_NM']
            
            # 1. 수기 매핑 테이블 확인
            if original_name in manual_mapping:
                search_name = manual_mapping[original_name]
            else:
                search_name = original_name
                
            # 2. 정규화 및 점수 조회
            norm_name = normalize_name(search_name)
            return if_db.get(y, {}).get(norm_name, 0)

        df['IF_Score'] = df.apply(get_score, axis=1)
        
        # 4. 상위 70% 추출
        print("✂️ [연도 x 중분류] 별 상위 70% 필터링 시작...")
        
        filtered_results = []
        groups = df.groupby(['temp_Year', 'NODE_CLSS_02'])
        
        for (year, clss), group in groups:
            total_count = len(group)
            
            # 상위 70% 설정
            target_count = int(total_count * 0.7)
            
            if target_count == 0:
                continue

            # 정렬: IF 점수 내림차순 -> NODE_ID 오름차순
            sorted_group = group.sort_values(by=['IF_Score', 'NODE_ID'], ascending=[False, True])
            
            # 자르기
            top_70 = sorted_group.head(target_count)
            filtered_results.append(top_70)

        # 5. 저장 (수정됨: NODE_LIST 키 포함 및 URL 슬래시 문제 해결)
        if filtered_results:
            final_df = pd.concat(filtered_results)
            
            # 원본 데이터 형태 유지를 위해 임시 컬럼(temp_Year) 삭제
            if 'temp_Year' in final_df.columns:
                final_df = final_df.drop(columns=['temp_Year'])
            
            output_json = 'SSU_Datathon2025_공학분야_62199_Top70.json'
            
            # 1. DataFrame을 리스트 형태의 딕셔너리로 변환
            final_data_list = final_df.to_dict(orient='records')
            
            # 2. 스크린샷과 동일하게 "NODE_LIST" 키로 감싸기
            final_output = {"NODE_LIST": final_data_list}
            
            # 3. json.dump로 저장 (URL 깔끔하게 유지)
            with open(output_json, 'w', encoding='utf-8') as f:
                json.dump(final_output, f, ensure_ascii=False, indent=4)
            
            print("\n" + "="*50)
            print(f"🎉 저장 완료! (구조: {{'NODE_LIST': [...]}})")
            print(f"- 원본 데이터: {len(df)}건")
            print(f"- 추출 데이터: {len(final_df)}건 (상위 70%)")
            print(f"- 파일명: {output_json}")
            print("="*50)
            
        else:
            print("조건에 맞는 논문이 없습니다.")

    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    main()

📂 인용지수 데이터베이스 구축 중...
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133824, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 132288, actual size 512
WARNING *** OLE2 stream 'SSCS': expected size 133312, actual size 512
📂 논문 데이터 로드 중... (SSU_Datathon2025_공학분야_62199_Increase_KYWD_Filtered.json)
📊 보정된 매핑 테이블을 사용하여 IF 점수 계산 중...
✂️ [연도 x 중분류] 별 상위 70% 필터링 시작...

🎉 저장 완료! (구조: {'NODE_LIST': [...]})
- 원본 데이터: 51163건
- 추출 데이터: 35790건 (상위 70%)
- 파일명: SSU_Datathon2025_공학분야_62199_Top70.json


In [3]:
# KYWD 필드 NAN -> "" 으로 변환

import json
import pandas as pd

# -----------------------------------------------------------
# 파일명 설정 (기존에 만드신 파일명과, 새로 저장할 파일명)
# -----------------------------------------------------------
input_file = 'top_70_percent_corrected.json'  # 수정할 대상 파일
output_file = 'top_70_percent_final.json'     # 최종 완성 파일

def clean_json_file():
    print(f"📂 파일 읽는 중... ({input_file})")
    
    try:
        # 1. JSON 파일 읽기
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # 데이터 구조 확인 (NODE_LIST 키가 있는지, 아니면 리스트 자체인지)
        if isinstance(data, dict) and "NODE_LIST" in data:
            target_list = data["NODE_LIST"]
        elif isinstance(data, list):
            target_list = data
        else:
            print("❌ 데이터 구조를 인식할 수 없습니다.")
            return

        # 2. DataFrame으로 변환하여 일괄 처리
        df = pd.DataFrame(target_list)

        # 3. 핵심 수정: NaN(결측치)를 빈 문자열("")로 변경
        #    (URL 슬래시 문제는 아래 json.dump가 자동으로 해결해줍니다)
        df = df.fillna("")
        
        print(f"📊 데이터 처리 중... (NaN 제거 및 URL 형식 보정)")

        # 4. 저장할 구조 만들기
        final_list = df.to_dict(orient='records')
        final_output = {"NODE_LIST": final_list}

        # 5. 최종 저장 (URL 슬래시 / 유지)
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(final_output, f, ensure_ascii=False, indent=4)

        print("\n" + "="*50)
        print(f"🎉 변환 완료!")
        print(f"1. NaN(결측치) -> \"\" (공란) 변경 완료")
        print(f"2. URL 슬래시(\\/) -> / (정상) 변경 완료")
        print(f"📂 저장된 파일: {output_file}")
        print("="*50)

    except FileNotFoundError:
        print(f"❌ 오류: '{input_file}' 파일을 찾을 수 없습니다.")
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

if __name__ == "__main__":
    clean_json_file()

📂 파일 읽는 중... (top_70_percent_corrected.json)
📊 데이터 처리 중... (NaN 제거 및 URL 형식 보정)

🎉 변환 완료!
1. NaN(결측치) -> "" (공란) 변경 완료
2. URL 슬래시(\/) -> / (정상) 변경 완료
📂 저장된 파일: top_70_percent_final.json


In [10]:
import json
import pandas as pd
import math
import os

# 1. 파일 경로 설정
original_file = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD_Filtered.json'   # 원본 파일
extracted_file = 'SSU_Datathon2025_공학분야_62199_Top70.json'           # 추출된 파일 (파일명 확인 필요)

def verify_extraction():
    print("🔍 검증 작업을 시작합니다... (기준: 상위 70%)\n")

    # -------------------------------------------------------
    # 1. 원본 데이터 분석 (목표치 계산)
    # -------------------------------------------------------
    try:
        with open(original_file, 'r', encoding='utf-8') as f:
            data_org = json.load(f)
            
        # 구조 유연성 처리
        if isinstance(data_org, dict) and "NODE_LIST" in data_org:
            df_org = pd.DataFrame(data_org["NODE_LIST"])
        else:
            df_org = pd.DataFrame(data_org)
            
        # 연도 추출
        df_org['Year'] = df_org['PBSH'].astype(str).str.strip().str[:4]
        
        # 그룹별 전체 개수 카운트
        org_stats = df_org.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Total_Count')
        
        # 목표 개수(Target) 계산: 전체 * 0.7 (소수점 내림)
        org_stats['Target_Count'] = (org_stats['Total_Count'] * 0.7).astype(int)
        
        print(f"✅ 원본 데이터 로드 완료 ({len(df_org)}건)")

    except Exception as e:
        print(f"❌ 원본 파일 로드 실패: {e}")
        return

    # -------------------------------------------------------
    # 2. 추출된 데이터 분석 (수정됨: 구조에 맞춰 로드)
    # -------------------------------------------------------
    try:
        # [수정] pd.read_json 대신 json.load 사용 후 리스트 접근
        with open(extracted_file, 'r', encoding='utf-8') as f:
            data_ext = json.load(f)
            
        # NODE_LIST 키가 있는지 확인하여 DataFrame 생성
        if isinstance(data_ext, dict) and "NODE_LIST" in data_ext:
            df_ext = pd.DataFrame(data_ext["NODE_LIST"])
        elif isinstance(data_ext, list):
            df_ext = pd.DataFrame(data_ext)
        else:
            print("❌ 추출 파일의 JSON 구조를 인식할 수 없습니다.")
            return

        # 연도 추출
        if 'Year' not in df_ext.columns:
             # PBSH가 존재하는지 확인
             if 'PBSH' in df_ext.columns:
                 df_ext['Year'] = df_ext['PBSH'].astype(str).str.strip().str[:4]
             else:
                 print("❌ 추출 파일에 'PBSH' 컬럼이 없습니다.")
                 return

        # 그룹별 추출 개수 카운트
        ext_stats = df_ext.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Actual_Count')
        
        print(f"✅ 추출 데이터 로드 완료 ({len(df_ext)}건)")

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {extracted_file}")
        return
    except Exception as e:
        print(f"❌ 추출 파일 로드 실패: {e}")
        return

    # -------------------------------------------------------
    # 3. 비교 검증 (Merge & Compare)
    # -------------------------------------------------------
    print("\n📊 검증 결과 집계 중...")
    
    # 원본 통계에 추출 통계 병합 (Left Join)
    merged = pd.merge(org_stats, ext_stats, on=['Year', 'NODE_CLSS_02'], how='left')
    
    # NaN은 0으로 채움
    merged['Actual_Count'] = merged['Actual_Count'].fillna(0).astype(int)
    
    # 검증: 목표치와 실제치가 같은가?
    merged['Is_Correct'] = merged['Target_Count'] == merged['Actual_Count']
    
    # -------------------------------------------------------
    # 4. 결과 리포트 출력
    # -------------------------------------------------------
    print("\n" + "="*80)
    print(f"{'Year':<6} | {'Category':<15} | {'Total':<7} | {'Target(70%)':<11} | {'Actual':<7} | {'Status'}")
    print("="*80)
    
    all_pass = True
    
    for idx, row in merged.iterrows():
        year = row['Year']
        clss = row['NODE_CLSS_02']
        total = row['Total_Count']
        target = row['Target_Count']
        actual = row['Actual_Count']
        is_correct = row['Is_Correct']
        
        if not is_correct:
            all_pass = False
            status = "❌ Mismatch"
        else:
            status = "✅ Pass"
            
        print(f"{year:<6} | {clss:<15} | {total:<7} | {target:<11} | {actual:<7} | {status}")

    sum_total = merged['Total_Count'].sum()
    sum_target = merged['Target_Count'].sum()
    sum_actual = merged['Actual_Count'].sum()

    print("-" * 80)
    print(f"{'TOTAL':<6} | {'ALL':<15} | {sum_total:<7} | {sum_target:<11} | {sum_actual:<7} | -")
    print("="*80)
    
    if all_pass:
        print("\n🎉 완벽합니다! 모든 그룹에서 정확히 상위 70% 개수만큼 추출되었습니다.")
    else:
        print("\n⚠️ 일부 그룹에서 개수가 일치하지 않습니다. 위 표를 확인해주세요.")
        print("(참고: 원본 그룹의 데이터 수가 매우 적을 경우, 반올림/내림 차이가 있을 수 있습니다.)")

if __name__ == "__main__":
    verify_extraction()

🔍 검증 작업을 시작합니다... (기준: 상위 70%)

✅ 원본 데이터 로드 완료 (51163건)
✅ 추출 데이터 로드 완료 (35790건)

📊 검증 결과 집계 중...

Year   | Category        | Total   | Target(70%) | Actual  | Status
2021   | 건축공학            | 1727    | 1208        | 1208    | ✅ Pass
2021   | 공학 일반           | 1122    | 785         | 785     | ✅ Pass
2021   | 기계공학            | 2100    | 1470        | 1470    | ✅ Pass
2021   | 기타 공학           | 369     | 258         | 258     | ✅ Pass
2021   | 산업공학            | 263     | 184         | 184     | ✅ Pass
2021   | 재료·에너지공학        | 241     | 168         | 168     | ✅ Pass
2021   | 전기전자공학          | 3192    | 2234        | 2234    | ✅ Pass
2021   | 조선해양공학          | 129     | 90          | 90      | ✅ Pass
2021   | 컴퓨터학            | 1159    | 811         | 811     | ✅ Pass
2021   | 화학공학            | 244     | 170         | 170     | ✅ Pass
2022   | 건축공학            | 1648    | 1153        | 1153    | ✅ Pass
2022   | 공학 일반           | 1084    | 758         | 758     | ✅ Pass
2022   | 기계공학     